# Shihua-VL：InternVL2.5-2B 基线复现

**前置条件：带 T4 GPU（16GB 显存）的笔记本运行时、Python 3、可联网，预留约 30 分钟。** 在所用笔记本环境中选择 T4 GPU 运行时。依赖与模型下载速度会影响总耗时。

本 notebook 仅运行 **InternVL2.5-2B（bf16）**，复现公开审计集的 20 件作品、60 题。沿用 runner 的原图动态切块（max_num=12，附缩略图）、贪心解码和 768 的输出长度上限。

公开仓库：https://github.com/Qi-520-QWQ/Shihua-VL 。首个代码 cell 会用 git clone 获取仓库。**若仓库尚未公开，改从本地上传仓库 zip，解压到工作根目录下的 `Shihua-VL` 后继续。** 可通过文件上传功能上传 `Shihua-VL.zip`；首个代码 cell 也会自动解压并整理目录。zip 可直接包含仓库文件，或包含一个仓库顶层文件夹。最终应有 `<工作根目录>/Shihua-VL/data/audit_v0/public_package/`。工作根目录默认为当前工作目录，可用环境变量 `SHIHUA_WORK_ROOT` 覆盖。

依次运行代码 cell。环境安装若提示重启，重启运行时后，再从首个代码 cell 重跑。词表补丁必须打印 **VALIDATED**，图像必须全部通过 SHA-256 校验，再运行两题 Smoke 和全量。遇到错误先处理该 cell 的报错。

结果默认逐题追加到 `<工作根目录>/Shihua-VL/data/audit_v0/public_package/predictions/internvl2_5_2b_reproduced.jsonl`。同一运行时中重跑全量 cell 即可断点续跑：自动跳过成功题，重试失败题。最后通过**下载** cell 保存结果；也可在中途运行下载 cell 保存已有进度。


In [ ]:
# 获取公开仓库；已解压的仓库可直接继续。
# 工作根目录：默认当前工作目录，可用环境变量 SHIHUA_WORK_ROOT 覆盖。
import os
from pathlib import Path
import shutil
import zipfile

WORK_ROOT = Path(os.environ.get("SHIHUA_WORK_ROOT", str(Path.cwd()))).expanduser()
repo_dir = WORK_ROOT / "Shihua-VL"
package_dir = repo_dir / "data/audit_v0/public_package"
if not package_dir.is_dir():
    !git clone --depth 1 https://github.com/Qi-520-QWQ/Shihua-VL {repo_dir}

repo_zip = WORK_ROOT / "Shihua-VL.zip"
if not package_dir.is_dir() and repo_zip.is_file():
    unpack_dir = WORK_ROOT / "shihua_vl_uploaded"
    with zipfile.ZipFile(repo_zip) as archive:
        archive.extractall(unpack_dir)
    candidates = [unpack_dir] + sorted(p for p in unpack_dir.iterdir() if p.is_dir())
    for candidate in candidates:
        if (candidate / "data/audit_v0/public_package").is_dir():
            shutil.copytree(candidate, repo_dir, dirs_exist_ok=True)
            break

assert package_dir.is_dir(), f"请上传仓库 zip 并解压到 {repo_dir}，再重跑本 cell。"
for relative_path in ("scripts/run_audit_t4.py", "scripts/patch_tokenizer_null.py",
                      "data/questions.jsonl", "data/items.jsonl"):
    assert (package_dir / relative_path).is_file(), f"缺少公开包文件：{relative_path}"
%cd {package_dir}


## 已知故障史

1. transformers 5.x 与 InternVL2.5 自定义建模代码不兼容，因此固定使用 4.49.0。
2. tokenizer.model 含空字符 piece，sentencepiece 0.2.x 拒载；使用补丁脚本将 **NUL→U+E000**，词元 ID 保持不变。


In [ ]:
# 环境：使用环境自带的 GPU 版 PyTorch。
!pip install -q "transformers==4.49.0" accelerate bitsandbytes
!pip install -q "sentencepiece>=0.2,<0.3" protobuf timm einops pillow
import sys
import torch
import transformers

print('Python', sys.version.split()[0])
print('transformers', transformers.__version__)
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert transformers.__version__.split('.')[0] == '4', 'transformers 主版本必须为 4；请重启运行时后重跑。'
assert transformers.__version__ == '4.49.0', '安装后若版本未更新，请使用 Runtime → Restart session，再从开头重跑。'
assert sys.version_info.major == 3, '需要 Python 3。'
assert torch.cuda.is_available(), '请在所用笔记本环境中选择 T4 GPU 运行时。'
print('GPU', torch.cuda.get_device_name(0))
assert 'T4' in torch.cuda.get_device_name(0), '本复现需要 T4 GPU。'


In [ ]:
# 准备独立复现输出：runner 的固定文件名通过软链接逐题写入新文件。
import json

tokenizer_validated = False
images_validated = False
smoke_validated = False
with (package_dir / 'data/questions.jsonl').open(encoding='utf-8') as handle:
    questions = [json.loads(line) for line in handle if line.strip()]
assert len(questions) == 60, f'题数应为 60，实际为 {len(questions)}。'
assert len({q['id'] for q in questions}) == 60, '题目 ID 必须唯一。'
print('questions:', len(questions))

work_dir = WORK_ROOT / "shihua_vl_2b_reproduction"
work_dir.mkdir(parents=True, exist_ok=True)
predictions_dir = package_dir / 'predictions'
predictions_dir.mkdir(exist_ok=True)
reproduced_path = predictions_dir / 'internvl2_5_2b_reproduced.jsonl'
reproduced_manifest_path = predictions_dir / 'internvl2_5_2b_reproduced_manifest.json'
for runner_name, destination in (
    ('internvl2_5_2b.jsonl', reproduced_path),
    ('internvl2_5_2b_manifest.json', reproduced_manifest_path),
):
    alias = work_dir / runner_name
    if alias.is_symlink():
        assert alias.resolve() == destination.resolve(), f'输出链接目标不一致：{alias}'
    elif alias.exists():
        raise FileExistsError(f'独立输出位置已有普通文件，请检查：{alias}')
    else:
        alias.symlink_to(destination)
print('复现结果：', reproduced_path)


In [ ]:
# 词表补丁：仅处理 2B；必须打印 VALIDATED 才能继续。
import io
from contextlib import redirect_stdout

tokenizer_validated = False
patch_log = io.StringIO()
try:
    with redirect_stdout(patch_log):
        %run scripts/patch_tokenizer_null.py --models 2b --questions data/questions.jsonl --out-dir {work_dir}
finally:
    print(patch_log.getvalue())
assert '2b: VALIDATED ' in patch_log.getvalue(), '补丁未打印 VALIDATED，请先处理本 cell 的报错。'
with (work_dir / 'tokenizer_patch_manifest.json').open(encoding='utf-8') as handle:
    patch_manifest = json.load(handle)
patch_result = patch_manifest['models']['2b']
assert patch_result.get('patched') is True, '补丁目录尚未生成。'
assert patch_result.get('validated_questions') == 60, '补丁必须通过全部 60 题校验。'
tokenizer_dir = work_dir / 'tokenizer_patched_2b'
assert (tokenizer_dir / 'tokenizer.model').is_file(), '缺少补丁词表。'
tokenizer_validated = True
print('VALIDATED：补丁 tokenizer 已通过 60 题检查。')


In [ ]:
# 下载 20 张原图到 data/images/；已存在且 SHA-256 一致则跳过。
import hashlib
from urllib.request import Request, urlopen

images_validated = False
with Path('data/items.jsonl').open(encoding='utf-8') as handle:
    items = [json.loads(line) for line in handle if line.strip()]
assert len(items) == 20, f'作品数应为 20，实际为 {len(items)}。'
assert len({item['artwork_id'] for item in items}) == 20, '作品 ID 必须唯一。'
assert {item['artwork_id'] for item in items} == {q['artwork_id'] for q in questions}, '题目与图像目录不匹配。'
images_dir = Path('data/images')
images_dir.mkdir(parents=True, exist_ok=True)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

for index, item in enumerate(items, 1):
    artwork_id = item['artwork_id']
    expected = item['source']['sha256'].lower()
    image_path = images_dir / f'{artwork_id}.jpg'
    if image_path.is_file() and file_sha256(image_path) == expected:
        print(f'[{index}/20] {artwork_id}: SKIP SHA256_OK')
        continue
    partial_path = image_path.with_suffix('.jpg.part')
    try:
        request = Request(item['source']['image_url'],
                          headers={'User-Agent': 'Shihua-VL-public-reproduction/1.0'})
        with urlopen(request, timeout=120) as response, partial_path.open('wb') as output:
            shutil.copyfileobj(response, output)
        actual = file_sha256(partial_path)
        assert actual == expected, f'{artwork_id}: SHA-256 不一致，期望 {expected}，实际 {actual}'
        partial_path.replace(image_path)
        print(f'[{index}/20] {artwork_id}: DOWNLOADED SHA256_OK')
    finally:
        partial_path.unlink(missing_ok=True)
images_validated = True
print('图像校验通过：20/20 SHA-256 一致。')


In [ ]:
# Smoke：InternVL2.5-2B（bf16），先跑 2 题。
smoke_validated = False
assert tokenizer_validated and images_validated, '先完成词表与全部图像校验。'
%cd {package_dir}
%run scripts/run_audit_t4.py --model 2b --limit 2 --tokenizer-dir {work_dir}/tokenizer_patched_2b --bundle-dir data --out-dir {work_dir} --mirror-dir "" --max-num 12
assert reproduced_path.is_file(), 'Smoke 尚未生成复现结果，请检查上方报错。'
with reproduced_path.open(encoding='utf-8') as handle:
    smoke_records = [json.loads(line) for line in handle if line.strip()]
smoke_done_ids = {r['id'] for r in smoke_records if r.get('answer') and not r.get('error')}
assert len(smoke_done_ids) >= 2, 'Smoke 未完成两条成功答案；先处理本 cell 的错误，再重跑。'
smoke_validated = True
print('SMOKE_OK：已有', len(smoke_done_ids), '题成功。')


In [ ]:
# 全量：60 题；重跑本 cell 即可续跑，成功题跳过、失败题重试。
assert tokenizer_validated and images_validated and smoke_validated, '先完成校验与 Smoke。'
%cd {package_dir}
%run scripts/run_audit_t4.py --model 2b --tokenizer-dir {work_dir}/tokenizer_patched_2b --bundle-dir data --out-dir {work_dir} --mirror-dir "" --max-num 12


In [ ]:
# 汇总：每题取最新记录；平均耗时和平均峰值显存只统计成功题。
from statistics import fmean

assert reproduced_path.is_file(), '请先运行 Smoke 或全量 cell。'
with reproduced_path.open(encoding='utf-8') as handle:
    records = [json.loads(line) for line in handle if line.strip()]
latest = {}
for record in records:
    latest[record['id']] = record
expected_ids = {q['id'] for q in questions}
assert set(latest).issubset(expected_ids), '结果包含公开审计集之外的题目。'
ok = [r for r in latest.values() if r.get('answer') and not r.get('error')]
failed = len(latest) - len(ok)
seconds = [r['seconds'] for r in ok if r.get('seconds') is not None]
memory = [r['peak_mem_gb'] for r in ok if r.get('peak_mem_gb') is not None]
print('结果：', reproduced_path.relative_to(package_dir))
print(f'questions={len(expected_ids)} attempted={len(latest)} ok={len(ok)} failed={failed} unattempted={len(expected_ids) - len(latest)}')
print(f'追加记录={len(records)}，其中历史重试记录={len(records) - len(latest)}')
print(f'平均耗时：{fmean(seconds):.1f} 秒' if seconds else '平均耗时：N/A')
print(f'平均峰值显存：{fmean(memory):.2f} GiB' if memory else '平均峰值显存：N/A')


In [ ]:
# 下载：保存复现 JSONL 和本次运行清单；也可中途保存进度。
# 在 Jupyter 环境中显示下载链接；其他环境直接打印文件路径。
assert reproduced_path.is_file(), "请先运行 Smoke 或全量 cell。"
try:
    from IPython.display import FileLink, display
    display(FileLink(str(reproduced_path)))
    if reproduced_manifest_path.is_file():
        display(FileLink(str(reproduced_manifest_path)))
except Exception:
    pass
print("复现结果：", reproduced_path)
if reproduced_manifest_path.is_file():
    print("运行清单：", reproduced_manifest_path)


In [ ]:
# 可选：镜像。默认关闭；需要时设置环境变量 SHIHUA_MIRROR_DIR
# 为目标目录，再运行本 cell 复制已有结果。
import os
mirror_dir = Path(os.environ.get("SHIHUA_MIRROR_DIR", ""))
if mirror_dir:
    assert reproduced_path.is_file(), "请先生成复现结果。"
    mirror_dir.mkdir(parents=True, exist_ok=True)
    for source in (reproduced_path, reproduced_manifest_path):
        if source.is_file():
            shutil.copy2(source, mirror_dir / source.name)
    print("可选镜像已保存：", mirror_dir)
else:
    print("可选镜像未启用（未设置 SHIHUA_MIRROR_DIR）。")
